# LSTM PM2.5 Prediction - Demo with Synthetic Data

This is a demonstration notebook that shows how the LSTM model works using synthetic data.
It's useful for testing the model architecture without requiring the full AQS database.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

print("TensorFlow version:", tf.__version__)
np.random.seed(42)
tf.random.set_seed(42)

## Generate Synthetic PM2.5 Data

In [ ]:
# Generate synthetic hourly PM2.5 data for one year
n_hours = 365 * 24
dates = pd.date_range('2020-01-01', periods=n_hours, freq='h')

# Create realistic PM2.5 pattern with:
# - Daily cycle (higher during day)
# - Weekly cycle (higher on weekdays)
# - Seasonal trend (higher in winter)
# - Random noise

hours = np.arange(n_hours)
pm25 = (
    15.0 +  # Baseline
    5.0 * np.sin(2 * np.pi * hours / 24) +  # Daily cycle
    3.0 * np.sin(2 * np.pi * hours / (24 * 7)) +  # Weekly cycle
    8.0 * np.sin(2 * np.pi * hours / (24 * 365) + np.pi) +  # Seasonal (winter high)
    np.random.normal(0, 2, n_hours)  # Random noise
)

# Ensure non-negative values
pm25 = np.maximum(pm25, 0)

df = pd.DataFrame({
    'datetime': dates,
    'Sample Measurement': pm25
})

print(f"Generated {len(df)} hourly PM2.5 measurements")
print(f"\nStatistics:")
print(df['Sample Measurement'].describe())

# Plot the data
plt.figure(figsize=(15, 5))
plt.plot(df['datetime'], df['Sample Measurement'], alpha=0.7, linewidth=0.5)
plt.xlabel('Date')
plt.ylabel('PM2.5 (µg/m³)')
plt.title('Synthetic PM2.5 Data for One Year')
plt.grid(True, alpha=0.3)
plt.show()

## Add Time Features

In [ ]:
# Extract time features
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month

# Cyclical encoding
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# For this demo, we'll use trust = 1.0 for all data (no missing values)
df['trust'] = 1.0

print("Time features added:")
print(df.columns.tolist())
print("\nFirst few rows:")
print(df.head())

## Prepare Data for LSTM

In [ ]:
# Select features
feature_cols = ['Sample Measurement', 'trust', 'hour_sin', 'hour_cos',
                'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend']

features = df[feature_cols].values

# Normalize
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features)

print(f"Feature matrix shape: {features_scaled.shape}")

In [ ]:
# Create sequences
def create_sequences(data, n_steps_in=5, n_steps_out=5):
    X, y = [], []
    for i in range(len(data) - n_steps_in - n_steps_out + 1):
        X.append(data[i:i + n_steps_in])
        y.append(data[i + n_steps_in:i + n_steps_in + n_steps_out, 0])
    return np.array(X), np.array(y)

n_steps_in = 5
n_steps_out = 5

X, y = create_sequences(features_scaled, n_steps_in, n_steps_out)

print(f"Input sequences shape (X): {X.shape}")
print(f"Output sequences shape (y): {y.shape}")

In [ ]:
# Split data
train_size = int(0.7 * len(X))
val_size = int(0.15 * len(X))

X_train = X[:train_size]
y_train = y[:train_size]
X_val = X[train_size:train_size + val_size]
y_val = y[train_size:train_size + val_size]
X_test = X[train_size + val_size:]
y_test = y[train_size + val_size:]

print(f"Training set: X={X_train.shape}, y={y_train.shape}")
print(f"Validation set: X={X_val.shape}, y={y_val.shape}")
print(f"Test set: X={X_test.shape}, y={y_test.shape}")

## Build and Train LSTM Model

In [ ]:
# Build model
n_features = X_train.shape[2]

model = Sequential([
    LSTM(64, activation='tanh', return_sequences=True, input_shape=(n_steps_in, n_features)),
    Dropout(0.2),
    LSTM(32, activation='tanh', return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(n_steps_out)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

print("Model architecture:")
model.summary()

In [ ]:
# Train model
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

print("Training model...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

print("Training complete!")

## Visualize Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Model Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Training MAE')
axes[1].plot(history.history['val_mae'], label='Validation MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Model MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluate and Visualize Predictions

In [ ]:
# Make predictions
y_pred = model.predict(X_test)

# Denormalize
def denormalize_pm25(values, scaler, n_features):
    dummy = np.zeros((values.shape[0], n_features))
    dummy[:, 0] = values
    denorm = scaler.inverse_transform(dummy)
    return denorm[:, 0]

y_test_denorm = np.zeros_like(y_test)
y_pred_denorm = np.zeros_like(y_pred)

for t in range(n_steps_out):
    y_test_denorm[:, t] = denormalize_pm25(y_test[:, t], scaler, n_features)
    y_pred_denorm[:, t] = denormalize_pm25(y_pred[:, t], scaler, n_features)

print("Predictions denormalized")

In [ ]:
# Calculate metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Performance metrics for each prediction horizon:")
print("="*60)

for t in range(n_steps_out):
    mae = mean_absolute_error(y_test_denorm[:, t], y_pred_denorm[:, t])
    rmse = np.sqrt(mean_squared_error(y_test_denorm[:, t], y_pred_denorm[:, t]))
    r2 = r2_score(y_test_denorm[:, t], y_pred_denorm[:, t])
    
    print(f"Time step +{t+1}:")
    print(f"  MAE: {mae:.4f} µg/m³")
    print(f"  RMSE: {rmse:.4f} µg/m³")
    print(f"  R²: {r2:.4f}")
    print()

In [ ]:
# Plot example predictions
n_examples = 5
fig, axes = plt.subplots(n_examples, 1, figsize=(15, 3*n_examples))

for i in range(n_examples):
    idx = i * (len(y_test_denorm) // n_examples)
    time_steps = np.arange(1, n_steps_out + 1)
    
    axes[i].plot(time_steps, y_test_denorm[idx], 'o-', label='Actual', markersize=8, linewidth=2)
    axes[i].plot(time_steps, y_pred_denorm[idx], 's--', label='Predicted', markersize=8, linewidth=2)
    
    axes[i].set_xlabel('Hours Ahead')
    axes[i].set_ylabel('PM2.5 (µg/m³)')
    axes[i].set_title(f'Example {i+1}: 5-Hour Prediction')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xticks(time_steps)

plt.tight_layout()
plt.show()

In [ ]:
# Time series plot
n_show = min(200, len(y_test_denorm))

fig, axes = plt.subplots(2, 1, figsize=(20, 10))

axes[0].plot(y_test_denorm[:n_show, 0], label='Actual', alpha=0.7, linewidth=2)
axes[0].plot(y_pred_denorm[:n_show, 0], label='Predicted', alpha=0.7, linewidth=2)
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('PM2.5 (µg/m³)')
axes[0].set_title('1-Hour Ahead Predictions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(y_test_denorm[:n_show, 4], label='Actual', alpha=0.7, linewidth=2)
axes[1].plot(y_pred_denorm[:n_show, 4], label='Predicted', alpha=0.7, linewidth=2)
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('PM2.5 (µg/m³)')
axes[1].set_title('5-Hour Ahead Predictions')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (t, ax) in enumerate([(0, axes[0]), (2, axes[1]), (4, axes[2])]):
    ax.scatter(y_test_denorm[:, t], y_pred_denorm[:, t], alpha=0.5, s=10)
    ax.plot([y_test_denorm[:, t].min(), y_test_denorm[:, t].max()],
            [y_test_denorm[:, t].min(), y_test_denorm[:, t].max()],
            'r--', linewidth=2, label='Perfect Prediction')
    ax.set_xlabel('Actual PM2.5 (µg/m³)')
    ax.set_ylabel('Predicted PM2.5 (µg/m³)')
    ax.set_title(f'{t+1}-Hour Ahead Prediction')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

This demo shows that the LSTM model can successfully learn patterns in PM2.5 data:
- The model captures daily, weekly, and seasonal cycles
- Short-term predictions (1-2 hours) are more accurate than long-term
- The model architecture and training process work correctly

For real AQS data, use the `lstm_airnow2.ipynb` notebook which includes:
- Database connectivity
- Trust-based data filling
- More comprehensive visualizations
- Better handling of missing data